### ガウシアンナイーブベイズとは
- ベイズの定理を使ってクラスの事後確率を計算し、最も確率が高いクラスに分類する手法
- 全ての特徴量が互いに独立であるという強い仮定を置く
- 「ガウシアン」は、各特徴量がガウス分布（正規分布）に従うという仮定に由来している

||内容|
|----|----|
|モデル|ベイズの定理 + ガウス分布|
|仮定|特徴量間の独立性、各特徴量がガウス分布に従う|
|学習|クラスごとの事前確率・平均・分散を計算|
|予測|対数事後確率が最大のクラスを選ぶ|
|安定性|積ではなく対数和で計算|



In [1]:
# ライブラリでの実装

import numpy as np
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


In [2]:
# scratchでのガウシアンナイーブベイズの実装

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class GaussianNB:
    def __init__(self):
        self.classes = None
        self.priors = None
        self.means = None
        self.vars = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples, n_features = X.shape
        n_classes = len(self.classes)

        self.priors = np.zeros(n_classes)
        self.means  = np.zeros((n_classes, n_features))
        self.vars   = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.priors[i] = len(X_c) / n_samples
            self.means[i]  = X_c.mean(axis=0)
            self.vars[i]   = X_c.var(axis=0)

    def _log_likelihood(self, x, mean, var):
        return -0.5 * np.sum(np.log(2 * np.pi * var) + (x - mean) ** 2 / var)

    def _predict_one(self, x):
        log_posteriors = []
        for i, c in enumerate(self.classes):
            log_prior = np.log(self.priors[i])
            log_likelihood = self._log_likelihood(x, self.means[i], self.vars[i])
            log_posteriors.append(log_prior + log_likelihood)
        return self.classes[np.argmax(log_posteriors)]

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])


# 実行
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")


正解率: 1.0000


`fit` メソッド

- クラスごとに事前確率・平均・分散を計算して保存します。`X_c.var(axis=0)` で各特徴量の分散をまとめて求めています。

`_log_likelihood` メソッド

- ガウス分布の対数尤度をそのまま実装しています。`np.log(2 * np.pi * var)` の部分が正規化項、`(x - mean) ** 2 / var` の部分が指数項に対応しています。`np.sum` で全特徴量の対数尤度を足し合わせることが、独立仮定による積を対数で和に変換することに相当します。

`_predict_one` メソッド

- 各クラスの対数事後確率（対数事前確率 + 対数尤度）を計算し、最大のクラスを返します。